# Breakout 2B · Sprint 3 · the invoice line against its neighbours

The invoice lines as the invoices state them, and the accepted prices nearest to each (similarity computed for you by a text model), each in its own currency and on its own delivery terms. **V2 is worked.** You put every price on one basis (dollars, FOB, net of the discount), build the range, and read where the declared price sits.

*Published customs datasets record an irregularity on 1 to 8 lines in 100; in this lab the random draw finds 3 in 100.*

In [ ]:
# Run this cell first (click it, then Shift + Enter). It finds the lab data.
import hashlib, os
from pathlib import Path
import numpy as np, pandas as pd

CANDIDATES = [os.environ.get('WORKSHOP_DATA', ''), 'data', '../data', '/workspace/data', '.']
DATA = next(Path(c) for c in CANDIDATES if c and (Path(c) / 'tariff.csv').exists())
print('Data folder:', DATA)

def _h(v, nd):
    return hashlib.sha256(f'{round(float(v), nd):.{nd}f}'.encode()).hexdigest()[:16]
def _hs(v):
    return hashlib.sha256(str(v).strip().lower().encode()).hexdigest()[:16]
def check(name, value, expected, nd=None):
    if callable(value):
        try:
            value = value()
        except Exception:
            value = ...
    try:
        ok = value is not ... and (_hs(value) == expected if nd is None else _h(value, nd) == expected)
    except Exception:
        ok = False
    shown = value if nd is None or value is ... else f'{value:,.{nd}f}'
    print(('✓ ' if ok else '✗ not yet: ') + name + ' = ' + str(shown))


In [ ]:
inv = pd.read_csv(DATA / 'invoice_lines.csv')
nbrs = pd.read_csv(DATA / 'valuation_neighbours_invoice.csv')
fx = pd.read_csv(DATA / 'fx_rates.csv').set_index(['month', 'currency']).lcu_per_unit
wedge = pd.read_csv(DATA / 'tariff.csv').drop_duplicates('chapter').set_index('chapter').cif_fob_wedge
def to_usd_fob(price, month, currency, incoterm, chapter, discount=0.0):
    w = wedge[chapter]
    factor = {'FOB': 1, 'EXW': 1.03, 'CIF': 1 / (1 + w), 'CFR': 1 / (1 + 0.85 * w)}[incoterm]
    return price * (1 - discount) * fx[(month, currency)] / fx[(month, 'USD')] * factor

## Worked: V2

In [ ]:
x = inv.set_index('ref').loc['V2']
declared = to_usd_fob(x.unit_price, x.invoice_date[:7], x.currency, x.incoterm, int(str(x.hs6)[:2]), x.discount)
n = nbrs[nbrs.query_id == 'V2'].copy()
n['usd_fob'] = [to_usd_fob(p, a[:7], c, i, int(str(h)[:2])) for p, a, c, i, h in zip(n.unit_price, n.accepted_on, n.currency, n.incoterm, n.hs8)]
kept = n[n.similarity >= 0.85].usd_fob
median = kept.median(); spread = 1.4826 * (kept - median).abs().median()   # as in 1B: a robust standard deviation
print(f'declared {declared:.2f} · {len(kept)} neighbours · median {median:.2f} · middle half {kept.quantile(.25):.2f} to {kept.quantile(.75):.2f} · {declared / median:.0%} of the median · {(declared - median) / spread:.2f} spreads')

## YOUR TURN: V1, V3, V4, V5

In [ ]:
# YOUR TURN: the same for another line; change Q and run again
Q = 'V4'
declared = ...
kept = ...       # its neighbours above 0.85, priced in USD FOB
median = ...
spread = ...     # 1.4826 × the median distance from the median
distance = ...   # (declared - median) / spread

In [ ]:
E = {'V1': {'declared': '78cccaf975d6cd46', 'n': '624b60c58c9d8bfb', 'median': 'e2bb398a8a355b72', 'distance': '773a15ec53a8a09e'}, 'V3': {'declared': '9fe5bc9796ce9e87', 'n': '624b60c58c9d8bfb', 'median': '8cf01561c22512c1', 'distance': 'bc2375185cb38cc7'}, 'V4': {'declared': 'ed5ac39d7c67435b', 'n': '5f9c4ab08cac7457', 'median': '8cb153e35493dbf3', 'distance': 'a77df445604efb92'}, 'V5': {'declared': '7a38e667a7e115e2', 'n': '4b227777d4dd1fc6', 'median': 'cf6663cef165d8bf', 'distance': 'b27688d3505e042d'}}
check(Q + ': declared, USD FOB', lambda: declared, E[Q]['declared'], 2)
check(Q + ': neighbours above the floor', lambda: len(kept), E[Q]['n'], 0)
check(Q + ': median', lambda: median, E[Q]['median'], 2)
check(Q + ': distance in spreads', lambda: distance, E[Q]['distance'], 2)

## Decide

Question through the valuation procedure (under the middle half, and a field on the invoice explains why), indicator only, or nothing. Cite the field that decided. The range is a reason to ask, never a value.